# 01 — Supervised Fine-Tuning

Before applying DPO alignment, we first fine-tune distilgpt2 on the **chosen** bios from our preference dataset. This SFT step teaches the model the dating bio domain — format, tone, length — before we ask it to distinguish good from bad. Starting DPO from a domain-adapted model produces better alignment than starting from the raw base model, which has no concept of what a dating bio should look like.

In [ ]:
import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import subprocess, sys
repo_root = "/content/aipi590-challenge-2"
token = os.environ["GITHUB_TOKEN"]
if not os.path.exists(repo_root):
    subprocess.run(
        f"git clone https://{token}@github.com/jonasneves/aipi590-challenge-2.git {repo_root}",
        shell=True, check=True
    )

!pip install -q trl peft transformers datasets accelerate bitsandbytes

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Always reload colab_utils so a re-run picks up the latest version from the repo
import importlib, src.colab_utils as _cu
importlib.reload(_cu)
from src.colab_utils import prepare_notebook, ensure_requirements, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")

# Reload again after prepare_notebook pulls the latest code
importlib.reload(_cu)
from src.colab_utils import prepare_notebook, ensure_requirements, publish_artifacts

ensure_requirements(repo_root)


In [ ]:
from google.colab import files
import shutil

pref_path = paths["data"] / "preferences.json"
if not pref_path.exists():
    print("Upload preferences.json from the MyRight admin export...")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.move(fname, pref_path)
print(f"Loaded: {pref_path} ({pref_path.stat().st_size // 1024}KB)")

In [ ]:
from src.dataset import load_sft_dataset

dataset = load_sft_dataset(pref_path)
print(f"Train: {len(dataset['train'])} examples")
print(f"Val:   {len(dataset['val'])} examples")
print("\nSample:")
print(dataset["train"][0])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="/content/sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    max_length=256,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    processing_class=tokenizer,
)
trainer.train()

# Persist log history so the plot cell works even after a kernel hiccup
import json as _json
(paths["results"] / "sft_log_history.json").write_text(
    _json.dumps(trainer.state.log_history, indent=2)
)


In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained("/content/sft_model")
tokenizer.save_pretrained("/content/sft_model")
print("SFT model saved to /content/sft_model")

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path

log_history = json.loads((paths["results"] / "sft_log_history.json").read_text())
train_losses = [(e["step"], e["loss"]) for e in log_history if "loss" in e]
steps, losses = zip(*train_losses)

plt.figure(figsize=(8, 4))
plt.plot(steps, losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("SFT Training Loss")
plt.tight_layout()

chart_path = paths["results"] / "sft_loss.png"
plt.savefig(chart_path, dpi=150)
plt.show()

metrics_path = paths["results"] / "sft_metrics.json"
metrics_path.write_text(json.dumps({
    "final_loss": round(losses[-1], 4),
    "steps": len(steps),
}, indent=2))

publish_artifacts(
    [chart_path, metrics_path],
    "add sft training results",
    repo_root,
)
print("Published.")
